In [1]:
pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


##Data Imported to Database

In [3]:
import pandas as pd
import mysql.connector
from getpass import getpass

# csv file path -> table name (order matters if you have foreign keys)
files = {
    r"C:\Users\hp\Desktop\pe\products_cleaned.csv":   "products",
    r"C:\Users\hp\Desktop\pe\cleaned_user_N.csv":   "users",
    r"C:\Users\hp\Desktop\pe\sessions.csv":    "sessions",
    r"C:\Users\hp\Desktop\pe\interactions_cleaned.csv":      "interactions",
    r"C:\Users\hp\Desktop\pe\purchases_cleaned.csv": "purchases",
    r"C:\Users\hp\Desktop\pe\reviews.csv": "reviews",
}

conn = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password=getpass("MySQL password: "),
    database="E_Commerce_Customer_Intelligence"
)
cursor = conn.cursor()

for path, table in files.items():
    df = pd.read_csv(path)
    df = df.astype(object).where(pd.notnull(df), None)  # NaN -> NULL

    cols = ", ".join(f"`{c}`" for c in df.columns)
    placeholders = ", ".join(["%s"] * len(df.columns))
    sql = f"INSERT INTO `{table}` ({cols}) VALUES ({placeholders})"

    cursor.executemany(sql, df.values.tolist())
    conn.commit()
    print(f"{table}: {cursor.rowcount} rows inserted")

cursor.close()
conn.close()

MySQL password: ··········
products: 1000 rows inserted
users: 10000 rows inserted
sessions: 19315 rows inserted
interactions: 100000 rows inserted
purchases: 1737 rows inserted
reviews: 1253 rows inserted


##Data Fetching from database

In [5]:
import csv
import mysql.connector

# TODO: Replace 'YOUR_MYSQL_PASSWORD' with your actual local MySQL root password
conn = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password="Ak@shuniyal97",
    database="E_Commerce_Customer_Intelligence"
)

cursor = conn.cursor()

In [6]:
cursor = conn.cursor()

cursor.execute("SHOW TABLES")

for table in cursor.fetchall():
    print(table[0])

interactions
products
purchases
reviews
sessions
users


In [7]:
import pandas as pd

##Create the customer-level RFM features

In [40]:
import pandas as pd
import numpy as np
import mysql.connector

# Reuse your existing connection
# conn = your existing MySQL connection

users = pd.read_sql("SELECT * FROM users", conn)
products = pd.read_sql("SELECT * FROM products", conn)
sessions = pd.read_sql("SELECT * FROM sessions", conn)
interactions = pd.read_sql("SELECT * FROM interactions", conn)
purchases = pd.read_sql("SELECT * FROM purchases", conn)
reviews = pd.read_sql("SELECT * FROM reviews", conn)

# Check data sizes
for name, df in {
    "users": users,
    "products": products,
    "sessions": sessions,
    "interactions": interactions,
    "purchases": purchases,
    "reviews": reviews
}.items():
    print(f"{name}: {df.shape}")

print(purchases.head())

C:\Users\hp\AppData\Local\Temp\ipykernel_24468\3850821417.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  users = pd.read_sql("SELECT * FROM users", conn)
C:\Users\hp\AppData\Local\Temp\ipykernel_24468\3850821417.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  products = pd.read_sql("SELECT * FROM products", conn)
C:\Users\hp\AppData\Local\Temp\ipykernel_24468\3850821417.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sessions = pd.read_sql("SELECT * FROM sessions", conn)
C:\Users\hp\AppData\Local\Temp\ipykerne

users: (10000, 9)
products: (1000, 11)
sessions: (19315, 6)
interactions: (100000, 7)
purchases: (1737, 10)
reviews: (1253, 8)
                            purchase_id                              order_id  \
0  0006c1d8-fb62-4474-938d-deb7851ef456  441417d7-72b5-4552-96fb-e5016cd9da3c   
1  000b86cc-387b-4968-aaa5-7152296985d2  a0c9aebc-d053-4fde-a490-63b751053b0e   
2  002437e2-0f73-4074-882d-0112757a3f48  77b26ebc-9dab-4d79-9326-c3f8a62bdbcb   
3  0050835e-2114-4f21-9343-4c92f0c6fcd0  d037464b-9bc4-4896-8133-1ffec764a200   
4  008fb553-ebfb-4b32-887e-6d0a47562fe6  2e882203-0773-40a8-a969-2571d9a1b940   

                                user_id                            product_id  \
0  af9d7b98-e46c-42f0-8b5e-a878c97c81d8  460c1852-1fe9-441b-be3b-599d95c07fea   
1  74073ef0-27ce-4323-a134-ae96c9cd6dca  c1ab2206-9b50-4aaa-9446-f322d0976ea6   
2  154bad5e-fc84-41bb-a290-2d17a5c6c948  6e9b539a-77ad-4307-acb5-f558f8b5cc6f   
3  2024a6da-c90b-474f-9842-e7986277d43f  7b43bf88-22b8-4623-b7

C:\Users\hp\AppData\Local\Temp\ipykernel_24468\3850821417.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  purchases = pd.read_sql("SELECT * FROM purchases", conn)
C:\Users\hp\AppData\Local\Temp\ipykernel_24468\3850821417.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  reviews = pd.read_sql("SELECT * FROM reviews", conn)


In [74]:

purchases["order_date"] = pd.to_datetime(
    purchases["order_date"], errors="coerce"
)

purchases["quantity"] = pd.to_numeric(
    purchases["quantity"], errors="coerce"
)

purchases["unit_price"] = pd.to_numeric(
    purchases["unit_price"], errors="coerce"
)

purchases["total_amount"] = pd.to_numeric(
    purchases["total_amount"], errors="coerce"
)

print(purchases[[
    "user_id", "order_id", "order_date",
    "quantity", "unit_price", "total_amount"
]].dtypes)


user_id                    str
order_id                   str
order_date      datetime64[us]
quantity                 int64
unit_price             float64
total_amount           float64
dtype: object


In [75]:

purchase_data = purchases.dropna(
    subset=["user_id", "order_id", "order_date"]
).copy()

print("Original purchase rows:", len(purchases))
print("Valid purchase rows:", len(purchase_data))
print("Unique purchasing customers:", purchase_data["user_id"].nunique())


Original purchase rows: 1737
Valid purchase rows: 1737
Unique purchasing customers: 1287


In [76]:

orders = (
    purchase_data.groupby(
        ["user_id", "order_id"], as_index=False
    )
    .agg(
        order_date=("order_date", "max"),
        order_value=("total_amount", "sum"),
        order_quantity=("quantity", "sum")
    )
)

print("Unique customer orders:", orders["order_id"].nunique())
display(orders.head())


Unique customer orders: 1440


,user_id,order_id,order_date,order_value,order_quantity
0,000b86cc-387b-4968-aaa5-7152296985d2,fe8b06e0-e4ed-442c-ac3e-159d873663d4,2026-02-17 20:41:31,62.87,1
1,007e5e78-dff8-449a-8f0a-f59185cad907,c1c96648-8f73-4224-bcd9-50455bb31627,2024-08-12 21:12:01,138.45,2
2,00922246-25f1-4564-92c3-2a314a9c6226,a17e8f73-abd0-4589-9b17-34c26d09c51c,2023-07-07 20:27:06,105.54,1
3,0094df58-edfe-4ecf-b214-94180bb4116e,01722924-508b-4f42-bb2d-f0aec08dd0d8,2023-06-21 11:01:40,61.47,1
4,0094df58-edfe-4ecf-b214-94180bb4116e,8ed2eb81-ff6f-4656-99d7-98d29466741b,2023-06-06 21:40:02,89.87,2


In [77]:

as_of_date = pd.Timestamp("2026-06-01")

customer_purchase_features = (
    orders.groupby("user_id")
    .agg(
        last_purchase_date=("order_date", "max"),
        first_purchase_date=("order_date", "min"),
        frequency=("order_id", "nunique"),
        monetary_value=("order_value", "sum"),
        total_quantity=("order_quantity", "sum"),
        average_order_value=("order_value", "mean")
    )
    .reset_index()
)

# Recency: days since the last purchase
customer_purchase_features["recency"] = (
    as_of_date -
    customer_purchase_features["last_purchase_date"]
).dt.days

# Observed customer purchase period, in days
customer_purchase_features["customer_lifetime_days"] = (
    customer_purchase_features["last_purchase_date"]
    - customer_purchase_features["first_purchase_date"]
).dt.days + 1

# Purchase frequency: orders per day in the observed period
customer_purchase_features["purchase_frequency"] = (
    customer_purchase_features["frequency"]
    / customer_purchase_features["customer_lifetime_days"]
)

display(customer_purchase_features.head(10))


,user_id,last_purchase_date,first_purchase_date,frequency,monetary_value,total_quantity,average_order_value,recency,customer_lifetime_days,purchase_frequency
0,000b86cc-387b-4968-aaa5-7152296985d2,2026-02-17 20:41:31,2026-02-17 20:41:31,1,62.87,1,62.870,103,1,1.000000
1,007e5e78-dff8-449a-8f0a-f59185cad907,2024-08-12 21:12:01,2024-08-12 21:12:01,1,138.45,2,138.450,657,1,1.000000
2,00922246-25f1-4564-92c3-2a314a9c6226,2023-07-07 20:27:06,2023-07-07 20:27:06,1,105.54,1,105.540,1059,1,1.000000
3,0094df58-edfe-4ecf-b214-94180bb4116e,2023-06-21 11:01:40,2023-06-06 21:40:02,2,151.34,3,75.670,1075,15,0.133333
4,009dd6c6-a577-4e67-86e3-282903230157,2025-02-20 22:51:36,2025-01-16 07:33:36,2,76.32,3,38.160,465,36,0.055556
5,00b132c6-78c4-4c2c-982f-387f7516c0f1,2025-02-18 18:31:21,2025-02-18 18:31:21,1,69.22,1,69.220,467,1,1.000000
6,00be33ef-b050-4ac3-8f72-f509a73356c5,2025-08-20 12:38:17,2025-08-20 12:38:17,1,114.19,2,114.190,284,1,1.000000
7,011a7341-1f25-4cd0-9777-edabe4c60de7,2023-02-10 04:01:02,2023-02-10 04:01:02,1,121.26,3,121.260,1206,1,1.000000
8,011ac2e7-8641-4b9d-8f80-61a70f82b9ff,2024-01-26 09:22:36,2024-01-26 09:22:36,1,49.20,1,49.200,856,1,1.000000
9,011cb619-3e69-47bd-913c-670ecea4383c,2024-08-30 22:02:07,2024-08-24 09:59:19,2,141.93,7,70.965,639,7,0.285714


In [78]:

customer_feature_table = users[["user_id"]].drop_duplicates().merge(
    customer_purchase_features,
    on="user_id",
    how="left",
    validate="one_to_one"
)

print("Total registered customers:",
      customer_feature_table["user_id"].nunique())

print("Final feature table shape:",
      customer_feature_table.shape)

display(customer_feature_table.head(10))


Total registered customers: 10000
Final feature table shape: (10000, 10)


,user_id,last_purchase_date,first_purchase_date,frequency,monetary_value,total_quantity,average_order_value,recency,customer_lifetime_days,purchase_frequency
0,0000780a-2126-4e84-9622-42ce0ea9b17a,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,000aea63-23ad-42ab-98b8-3de2e33c624c,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,000b86cc-387b-4968-aaa5-7152296985d2,2026-02-17 20:41:31,2026-02-17 20:41:31,1.0,62.87,1.0,62.87,103.0,1.0,1.0
3,0015db27-c5fe-4846-ab49-599a3384d90d,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,001f3a0f-80f7-433e-86cd-9027ce848f27,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,003a6567-405b-4dfc-8786-956fa904f11c,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,004096e5-6294-414e-997d-b96f22c86b3e,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,004c385b-f83f-46e2-8523-694304a93d76,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,004eb82c-1643-4850-9738-fd2e2c81d645,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,0050835e-2114-4f21-9343-4c92f0c6fcd0,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [79]:

required_features = [
    "user_id",
    "recency",
    "frequency",
    "monetary_value",
    "purchase_frequency",
    "average_order_value"
]

display(customer_feature_table[required_features].head(10))

print("\nMissing values:")
print(customer_feature_table[required_features].isnull().sum())

print("\nCustomers with purchases:",
      customer_feature_table["frequency"].notna().sum())

print("Customers without purchases:",
      customer_feature_table["frequency"].isna().sum())


,user_id,recency,frequency,monetary_value,purchase_frequency,average_order_value
0,0000780a-2126-4e84-9622-42ce0ea9b17a,NaN,NaN,NaN,NaN,NaN
1,000aea63-23ad-42ab-98b8-3de2e33c624c,NaN,NaN,NaN,NaN,NaN
2,000b86cc-387b-4968-aaa5-7152296985d2,103.0,1.0,62.87,1.0,62.87
3,0015db27-c5fe-4846-ab49-599a3384d90d,NaN,NaN,NaN,NaN,NaN
4,001f3a0f-80f7-433e-86cd-9027ce848f27,NaN,NaN,NaN,NaN,NaN
5,003a6567-405b-4dfc-8786-956fa904f11c,NaN,NaN,NaN,NaN,NaN
6,004096e5-6294-414e-997d-b96f22c86b3e,NaN,NaN,NaN,NaN,NaN
7,004c385b-f83f-46e2-8523-694304a93d76,NaN,NaN,NaN,NaN,NaN
8,004eb82c-1643-4850-9738-fd2e2c81d645,NaN,NaN,NaN,NaN,NaN
9,0050835e-2114-4f21-9343-4c92f0c6fcd0,NaN,NaN,NaN,NaN,NaN



Missing values:
user_id                   0
recency                8713
frequency              8713
monetary_value         8713
purchase_frequency     8713
average_order_value    8713
dtype: int64

Customers with purchases: 1287
Customers without purchases: 8713


In [80]:

# Check one row per customer
print(
    "Duplicate customer IDs:",
    customer_feature_table["user_id"].duplicated().sum()
)

# Check feature values for customers with purchases
purchasing_customers = customer_feature_table[
    customer_feature_table["frequency"].notna()
]

print(
    "Customers with invalid negative recency:",
    (purchasing_customers["recency"] < 0).sum()
)

print(
    "Customers with frequency below 1:",
    (purchasing_customers["frequency"] < 1).sum()
)

print(
    "Customers with negative monetary value:",
    (purchasing_customers["monetary_value"] < 0).sum()
)

display(
    purchasing_customers[
        [
            "user_id",
            "recency",
            "frequency",
            "monetary_value",
            "purchase_frequency",
            "average_order_value"
        ]
    ].head(10)
)


Duplicate customer IDs: 0
Customers with invalid negative recency: 0
Customers with frequency below 1: 0
Customers with negative monetary value: 0


,user_id,recency,frequency,monetary_value,purchase_frequency,average_order_value
2,000b86cc-387b-4968-aaa5-7152296985d2,103.0,1.0,62.87,1.000000,62.870
14,007e5e78-dff8-449a-8f0a-f59185cad907,657.0,1.0,138.45,1.000000,138.450
18,00922246-25f1-4564-92c3-2a314a9c6226,1059.0,1.0,105.54,1.000000,105.540
19,0094df58-edfe-4ecf-b214-94180bb4116e,1075.0,2.0,151.34,0.133333,75.670
22,009dd6c6-a577-4e67-86e3-282903230157,465.0,2.0,76.32,0.055556,38.160
28,00b132c6-78c4-4c2c-982f-387f7516c0f1,467.0,1.0,69.22,1.000000,69.220
29,00be33ef-b050-4ac3-8f72-f509a73356c5,284.0,1.0,114.19,1.000000,114.190
44,011a7341-1f25-4cd0-9777-edabe4c60de7,1206.0,1.0,121.26,1.000000,121.260
45,011ac2e7-8641-4b9d-8f80-61a70f82b9ff,856.0,1.0,49.20,1.000000,49.200
46,011cb619-3e69-47bd-913c-670ecea4383c,639.0,2.0,141.93,0.285714,70.965


## Select customers who have purchase records

In [81]:
segmentation_data = customer_feature_table[
    customer_feature_table["frequency"].notna()
].copy()

print("Customers selected for segmentation:", len(segmentation_data))
print("Duplicate customer IDs:", segmentation_data["user_id"].duplicated().sum())

display(segmentation_data.head())

Customers selected for segmentation: 1287
Duplicate customer IDs: 0


,user_id,last_purchase_date,first_purchase_date,frequency,monetary_value,total_quantity,average_order_value,recency,customer_lifetime_days,purchase_frequency
2,000b86cc-387b-4968-aaa5-7152296985d2,2026-02-17 20:41:31,2026-02-17 20:41:31,1.0,62.87,1.0,62.87,103.0,1.0,1.000000
14,007e5e78-dff8-449a-8f0a-f59185cad907,2024-08-12 21:12:01,2024-08-12 21:12:01,1.0,138.45,2.0,138.45,657.0,1.0,1.000000
18,00922246-25f1-4564-92c3-2a314a9c6226,2023-07-07 20:27:06,2023-07-07 20:27:06,1.0,105.54,1.0,105.54,1059.0,1.0,1.000000
19,0094df58-edfe-4ecf-b214-94180bb4116e,2023-06-21 11:01:40,2023-06-06 21:40:02,2.0,151.34,3.0,75.67,1075.0,15.0,0.133333
22,009dd6c6-a577-4e67-86e3-282903230157,2025-02-20 22:51:36,2025-01-16 07:33:36,2.0,76.32,3.0,38.16,465.0,36.0,0.055556


In [82]:

segmentation_features = [
    "recency",
    "frequency",
    "monetary_value",
    "purchase_frequency",
    "average_order_value"
]

X = segmentation_data[segmentation_features].copy()

print("Missing values:")
print(X.isnull().sum())

print("\nFeature summary:")
display(X.describe().round(2))


Missing values:
recency                0
frequency              0
monetary_value         0
purchase_frequency     0
average_order_value    0
dtype: int64

Feature summary:


,recency,frequency,monetary_value,purchase_frequency,average_order_value
count,1287.00,1287.00,1287.00,1287.00,1287.00
mean,524.55,1.12,100.63,0.93,90.91
std,333.59,0.34,109.62,0.26,101.66
min,0.00,1.00,1.93,0.03,1.93
25%,253.00,1.00,36.12,1.00,34.76
50%,482.00,1.00,66.55,1.00,61.39
75%,793.50,1.00,122.01,1.00,107.41
max,1232.00,4.00,1486.54,2.00,1486.54


In [83]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled data shape:", X_scaled.shape)


Scaled data shape: (1287, 5)


In [84]:

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd

results = []

for k in range(2, 9):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X_scaled)

    results.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette_score": silhouette_score(X_scaled, labels)
    })

cluster_results = pd.DataFrame(results)

display(cluster_results.round(3))


,k,inertia,silhouette_score
0,2,4247.957,0.598
1,3,2867.102,0.553
2,4,2107.219,0.439
3,5,1693.353,0.449
4,6,1484.180,0.463
5,7,1329.257,0.383
6,8,1190.513,0.386


##Final K-Means

In [85]:
from sklearn.cluster import KMeans

best_k = 2

kmeans_model = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

segmentation_data["cluster"] = kmeans_model.fit_predict(X_scaled)

print("Cluster sizes:")
display(
    segmentation_data["cluster"]
    .value_counts()
    .sort_index()
    .rename("customer_count")
    .to_frame()
)

print("Cluster percentages:")
display(
    (
        segmentation_data["cluster"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
        .rename("percentage")
        .to_frame()
    )
)

Cluster sizes:


,customer_count
cluster,
0,132
1,1155


Cluster percentages:


,percentage
cluster,
0,10.26
1,89.74


In [86]:
cluster_profile = (
    segmentation_data
    .groupby("cluster")[segmentation_features]
    .mean()
    .round(2)
)

display(cluster_profile)

,recency,frequency,monetary_value,purchase_frequency,average_order_value
cluster,,,,,
0,468.67,2.05,170.63,0.25,82.77
1,530.94,1.01,92.63,1.01,91.84


##Interpretation
- Cluster 0 — Repeat Purchasers: These customers have placed an average of 2.05 orders and have a total average spending of 170.63. They tend to purchase more frequently and spend more overall compared to customers in Cluster 1.

- Cluster 1 — Mostly One-Time Purchasers: These customers have placed an average of 1.01 orders and have a total average spending of 92.63. This segment primarily consists of customers who have made only one purchase.

- Recency: Cluster 0 has a lower average recency of 468.67 days compared to 530.94 days for Cluster 1. This indicates that customers in Cluster 0 have purchased more recently, on average. However, the recency values for both clusters are relatively high, so these customers should not automatically be classified as currently active.

In [87]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd

# Remove purchase_frequency from clustering features
revised_features = [
    "recency",
    "frequency",
    "monetary_value",
    "average_order_value"
]

X_revised = segmentation_data[revised_features].copy()

# Scale the features
scaler_revised = StandardScaler()
X_revised_scaled = scaler_revised.fit_transform(X_revised)

# Test different cluster counts
results_revised = []

for k in range(2, 9):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X_revised_scaled)

    results_revised.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette_score": silhouette_score(
            X_revised_scaled, labels
        )
    })

cluster_results_revised = pd.DataFrame(results_revised)

display(cluster_results_revised.round(3))

,k,inertia,silhouette_score
0,2,3639.598,0.498
1,3,2506.779,0.545
2,4,1758.118,0.448
3,5,1349.618,0.460
4,6,1196.558,0.381
5,7,1040.365,0.386
6,8,900.491,0.389


In [89]:
best_k = 3

final_kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

segmentation_data["final_cluster"] = final_kmeans.fit_predict(
    X_revised_scaled
)

print("Final cluster sizes:")
display(
    segmentation_data["final_cluster"]
    .value_counts()
    .sort_index()
    .rename("customer_count")
    .to_frame()
)

print("Cluster percentages:")
display(
    (
        segmentation_data["final_cluster"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
        .rename("percentage")
        .to_frame()
)
)

Final cluster sizes:


,customer_count
final_cluster,
0,145
1,1003
2,139


Cluster percentages:


,percentage
final_cluster,
0,11.27
1,77.93
2,10.80


In [90]:
final_cluster_profile = (
    segmentation_data
    .groupby("final_cluster")[revised_features]
    .mean()
    .round(2)
)

display(final_cluster_profile)

,recency,frequency,monetary_value,average_order_value
final_cluster,,,,
0,459.92,2.05,164.52,79.94
1,530.19,1.00,61.99,61.99
2,551.33,1.01,312.77,311.01


- Cluster 0 — Repeat Purchasers: These customers have placed approximately 2 orders on average and have an average total spending of 164.52. They also have the lowest recency, meaning they purchased more recently than the other clusters on average.

- Cluster 1 — Low-Spending, One-Time Purchasers: These customers have placed approximately 1 order on average and have the lowest average total spending of 61.99. This group represents customers with relatively low purchase value.

- Cluster 2 — High-Value, One-Time Purchasers: These customers have placed approximately 1 order on average, but their average total spending is 312.77, the highest among all three clusters. Their average order value is also substantially higher at 311.01.

##Customer Churn Prediciton

Define the prediction period
We'll use:
- Feature cutoff date: December 3, 2025
- Prediction window: The following 180 days
- Churn = 1: No purchase during the 180-day prediction window
- Churn = 0: At least one purchase during that window

In [91]:
import pandas as pd
import numpy as np

# Ensure order dates are datetime
purchase_data["order_date"] = pd.to_datetime(
    purchase_data["order_date"],
    errors="coerce"
)

# Define the historical cutoff and prediction window
cutoff_date = pd.Timestamp("2025-12-03")
prediction_end_date = cutoff_date + pd.Timedelta(days=180)

print("Feature cutoff date:", cutoff_date.date())
print("Prediction window ends:", prediction_end_date.date())

print("Earliest purchase date:",
      purchase_data["order_date"].min())

print("Latest purchase date:",
      purchase_data["order_date"].max())

Feature cutoff date: 2025-12-03
Prediction window ends: 2026-06-01
Earliest purchase date: 2023-01-15 18:05:44
Latest purchase date: 2026-05-31 17:56:50


In [92]:
# Define historical data and future prediction window
historical_purchases = purchase_data[
    purchase_data["order_date"] <= cutoff_date
].copy()

future_purchases = purchase_data[
    (purchase_data["order_date"] > cutoff_date) &
    (purchase_data["order_date"] <= prediction_end_date)
].copy()

# Aggregate historical purchases at order level
historical_orders = (
    historical_purchases
    .groupby(["user_id", "order_id"], as_index=False)
    .agg(
        order_date=("order_date", "max"),
        order_value=("total_amount", "sum"),
        order_quantity=("quantity", "sum")
    )
)

# Create customer features using historical purchases only
churn_features = (
    historical_orders
    .groupby("user_id")
    .agg(
        last_purchase_date=("order_date", "max"),
        first_purchase_date=("order_date", "min"),
        frequency=("order_id", "nunique"),
        monetary_value=("order_value", "sum"),
        total_quantity=("order_quantity", "sum"),
        average_order_value=("order_value", "mean")
    )
    .reset_index()
)

churn_features["recency"] = (
    cutoff_date - churn_features["last_purchase_date"]
).dt.days

churn_features["customer_lifetime_days"] = (
    churn_features["last_purchase_date"]
    - churn_features["first_purchase_date"]
).dt.days + 1

# Calculate future purchase activity
future_customer_ids = future_purchases["user_id"].dropna().unique()

churn_features["churn"] = (
    ~churn_features["user_id"].isin(future_customer_ids)
).astype(int)

print("Customers with historical purchases:", len(churn_features))
print("\nChurn label counts:")
display(churn_features["churn"].value_counts().sort_index().to_frame("customer_count"))

print("\nChurn label percentages:")
display(
    (churn_features["churn"].value_counts(normalize=True) * 100)
    .round(2)
    .sort_index()
    .to_frame("percentage")
)

display(churn_features.head())

Customers with historical purchases: 1050

Churn label counts:


,customer_count
churn,
0,1
1,1049



Churn label percentages:


,percentage
churn,
0,0.1
1,99.9


,user_id,last_purchase_date,first_purchase_date,frequency,monetary_value,total_quantity,average_order_value,recency,customer_lifetime_days,churn
0,007e5e78-dff8-449a-8f0a-f59185cad907,2024-08-12 21:12:01,2024-08-12 21:12:01,1,138.45,2,138.45,477,1,1
1,00922246-25f1-4564-92c3-2a314a9c6226,2023-07-07 20:27:06,2023-07-07 20:27:06,1,105.54,1,105.54,879,1,1
2,0094df58-edfe-4ecf-b214-94180bb4116e,2023-06-21 11:01:40,2023-06-06 21:40:02,2,151.34,3,75.67,895,15,1
3,009dd6c6-a577-4e67-86e3-282903230157,2025-02-20 22:51:36,2025-01-16 07:33:36,2,76.32,3,38.16,285,36,1
4,00b132c6-78c4-4c2c-982f-387f7516c0f1,2025-02-18 18:31:21,2025-02-18 18:31:21,1,69.22,1,69.22,287,1,1


In [112]:
# Predict segments using the existing trained K-Means model
segmentation_data["predicted_cluster"] = final_kmeans.predict(
    X_revised_scaled
)

print("Customer segmentation predictions completed!")
print("\nCustomers in each segment:")
display(
    segmentation_data["predicted_cluster"]
    .value_counts()
    .sort_index()
)

display(segmentation_data.head())

Customer segmentation predictions completed!

Customers in each segment:


predicted_cluster
0     145
1    1003
2     139
Name: count, dtype: int64

,user_id,last_purchase_date,first_purchase_date,frequency,monetary_value,total_quantity,average_order_value,recency,customer_lifetime_days,purchase_frequency,cluster,final_cluster,predicted_cluster
2,000b86cc-387b-4968-aaa5-7152296985d2,2026-02-17 20:41:31,2026-02-17 20:41:31,1.0,62.87,1.0,62.87,103.0,1.0,1.000000,1,1,1
14,007e5e78-dff8-449a-8f0a-f59185cad907,2024-08-12 21:12:01,2024-08-12 21:12:01,1.0,138.45,2.0,138.45,657.0,1.0,1.000000,1,1,1
18,00922246-25f1-4564-92c3-2a314a9c6226,2023-07-07 20:27:06,2023-07-07 20:27:06,1.0,105.54,1.0,105.54,1059.0,1.0,1.000000,1,1,1
19,0094df58-edfe-4ecf-b214-94180bb4116e,2023-06-21 11:01:40,2023-06-06 21:40:02,2.0,151.34,3.0,75.67,1075.0,15.0,0.133333,0,0,0
22,009dd6c6-a577-4e67-86e3-282903230157,2025-02-20 22:51:36,2025-01-16 07:33:36,2.0,76.32,3.0,38.16,465.0,36.0,0.055556,0,0,0


In [113]:
segment_names = {
    0: "Repeat Buyers",
    1: "Low-Spending One-Time Buyers",
    2: "High-Spending One-Time Buyers"
}

segmentation_data["segment_name"] = (
    segmentation_data["predicted_cluster"]
    .map(segment_names)
)

display(
    segmentation_data[
        [
            "user_id",
            "recency",
            "frequency",
            "monetary_value",
            "average_order_value",
            "predicted_cluster",
            "segment_name"
        ]
    ].head(10)
)

,user_id,recency,frequency,monetary_value,average_order_value,predicted_cluster,segment_name
2,000b86cc-387b-4968-aaa5-7152296985d2,103.0,1.0,62.87,62.870,1,Low-Spending One-Time Buyers
14,007e5e78-dff8-449a-8f0a-f59185cad907,657.0,1.0,138.45,138.450,1,Low-Spending One-Time Buyers
18,00922246-25f1-4564-92c3-2a314a9c6226,1059.0,1.0,105.54,105.540,1,Low-Spending One-Time Buyers
19,0094df58-edfe-4ecf-b214-94180bb4116e,1075.0,2.0,151.34,75.670,0,Repeat Buyers
22,009dd6c6-a577-4e67-86e3-282903230157,465.0,2.0,76.32,38.160,0,Repeat Buyers
28,00b132c6-78c4-4c2c-982f-387f7516c0f1,467.0,1.0,69.22,69.220,1,Low-Spending One-Time Buyers
29,00be33ef-b050-4ac3-8f72-f509a73356c5,284.0,1.0,114.19,114.190,1,Low-Spending One-Time Buyers
44,011a7341-1f25-4cd0-9777-edabe4c60de7,1206.0,1.0,121.26,121.260,1,Low-Spending One-Time Buyers
45,011ac2e7-8641-4b9d-8f80-61a70f82b9ff,856.0,1.0,49.20,49.200,1,Low-Spending One-Time Buyers
46,011cb619-3e69-47bd-913c-670ecea4383c,639.0,2.0,141.93,70.965,0,Repeat Buyers


In [117]:
# Prepare data
segmentation_predictions = segmentation_data[
    [
        "user_id",
        "recency",
        "frequency",
        "monetary_value",
        "average_order_value",
        "predicted_cluster",
        "segment_name"
    ]
].copy()

# Convert any NaN values to None for SQL NULL compatibility
segmentation_predictions = segmentation_predictions.astype(object).where(pd.notnull(segmentation_predictions), None)

# Use the existing mysql.connector connection
cursor = conn.cursor()

# Drop table if it exists to mimic 'replace' behavior
cursor.execute("DROP TABLE IF EXISTS customer_segmentation_predictions")

# Create table schema
cursor.execute("""
CREATE TABLE customer_segmentation_predictions (
    user_id VARCHAR(255),
    recency FLOAT,
    frequency FLOAT,
    monetary_value FLOAT,
    average_order_value FLOAT,
    predicted_cluster INT,
    segment_name VARCHAR(255)
)
""")

# Construct batch insert query
cols = ", ".join(f"`{c}`" for c in segmentation_predictions.columns)
placeholders = ", ".join(["%s"] * len(segmentation_predictions.columns))
sql_insert = f"INSERT INTO customer_segmentation_predictions ({cols}) VALUES ({placeholders})"

# Execute batch insert
cursor.executemany(sql_insert, segmentation_predictions.values.tolist())
conn.commit()

print(f"Customer segmentation predictions saved to MySQL! ({cursor.rowcount} rows inserted)")
cursor.close()

Customer segmentation predictions saved to MySQL! (1287 rows inserted)


True

In [120]:
import pandas as pd

# Map cluster IDs to segment names based on your trained final_kmeans profiles
segment_names = {
    0: "Repeat Buyers",
    1: "Low-Spending One-Time Buyers",
    2: "High-Spending One-Time Buyers"
}

def predict_customer(recency, frequency, monetary_value, average_order_value):
    # Prepare the raw input features
    customer = pd.DataFrame([{
        "recency": recency,
        "frequency": frequency,
        "monetary_value": monetary_value,
        "average_order_value": average_order_value
    }])

    # Use the active scaler from your kernel ('scaler_revised') to transform the raw features
    customer_scaled = scaler_revised.transform(customer)

    # Use the active model from your kernel ('final_kmeans') to predict the cluster
    cluster = int(final_kmeans.predict(customer_scaled)[0])

    print("Predicted Cluster:", cluster)
    print("Customer Segment:", segment_names[cluster])

In [121]:
predict_customer(30, 5, 1500, 300)

Predicted Cluster: 0
Customer Segment: Repeat Buyers


In [124]:
import joblib

joblib.dump(final_kmeans, "customer_segmentation_model.joblib")
joblib.dump(scaler_revised, "customer_segmentation_scaler.joblib")

print("Model and scaler saved successfully!")

Model and scaler saved successfully!


In [126]:
import os

print("Folder:", os.getcwd())
print("Model exists:", os.path.exists("customer_segmentation_model.joblib"))
print("Scaler exists:", os.path.exists("customer_segmentation_scaler.joblib"))

Folder: C:\Windows\System32
Model exists: True
Scaler exists: True
